In [27]:
import os

In [28]:
%pwd

'd:\\Text-Summarization-NLP-Project'

In [29]:
os.chdir("../")

In [30]:
%pwd

'd:\\'

In [31]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path


In [32]:
import sys
sys.path.append("D:/Text-Summarization-NLP-Project/src")

In [33]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [34]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name = config.tokenizer_name
        )

        return data_transformation_config


In [35]:
! pip install datasets

  Using cached pyarrow-21.0.0-cp39-cp39-win_amd64.whl.metadata (3.4 kB)
Using cached pyarrow-21.0.0-cp39-cp39-win_amd64.whl (26.2 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 11.0.0
    Uninstalling pyarrow-11.0.0:
      Successfully uninstalled pyarrow-11.0.0


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 2.2.2 requires numpy<2, but you have numpy 2.0.2 which is incompatible.
mlflow 2.2.2 requires pyarrow<12,>=4.0.0, but you have pyarrow 21.0.0 which is incompatible.


In [36]:
!pip uninstall pyarrow -y

Found existing installation: pyarrow 21.0.0
Uninstalling pyarrow-21.0.0:
  Successfully uninstalled pyarrow-21.0.0


In [37]:
!pip install pyarrow==11.0.0 --force-reinstall --no-cache-dir

   ---------------------------------------- 0.0/20.6 MB ? eta -:--:--
   - -------------------------------------- 0.8/20.6 MB 5.6 MB/s eta 0:00:04
   ------- -------------------------------- 3.7/20.6 MB 11.5 MB/s eta 0:00:02
   ------------- -------------------------- 6.8/20.6 MB 13.1 MB/s eta 0:00:02
   ------------------- -------------------- 10.2/20.6 MB 13.9 MB/s eta 0:00:01
   ------------------------- -------------- 13.1/20.6 MB 13.9 MB/s eta 0:00:01
   ------------------------------- -------- 16.3/20.6 MB 14.0 MB/s eta 0:00:01
   ------------------------------------- -- 19.4/20.6 MB 14.2 MB/s eta 0:00:01
   ---------------------------------------- 20.6/20.6 MB 14.3 MB/s  0:00:01
   ---------------------------------------- 0.0/15.9 MB ? eta -:--:--
   ------- -------------------------------- 3.1/15.9 MB 15.3 MB/s eta 0:00:01
   --------------- ------------------------ 6.3/15.9 MB 15.4 MB/s eta 0:00:01
   ----------------------- ---------------- 9.4/15.9 MB 15.4 MB/s eta 0:00:01
 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.5.0 requires pyarrow>=21.0.0, but you have pyarrow 11.0.0 which is incompatible.
mlflow 2.2.2 requires numpy<2, but you have numpy 2.0.2 which is incompatible.
tensorflow-intel 2.13.0 requires numpy<=1.24.3,>=1.22, but you have numpy 2.0.2 which is incompatible.
tensorflow-intel 2.13.0 requires typing-extensions<4.6.0,>=3.6.6, but you have typing-extensions 4.15.0 which is incompatible.


In [38]:
import pyarrow
import pyarrow.parquet as pq

print(pyarrow.__version__)

11.0.0


In [39]:
import os
from textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

In [40]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)


    
    def convert_examples_to_features(self,example_batch):
        input_encodings = self.tokenizer(example_batch['dialogue'] , max_length = 1024, truncation = True )
        
        with self.tokenizer.as_target_tokenizer():
            target_encodings = self.tokenizer(example_batch['summary'], max_length = 128, truncation = True )
            
        return {
            'input_ids' : input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }
    

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched = True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir,"samsum_dataset"))

In [44]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-03-13 17:15:02,844: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-13 17:15:02,844: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-13 17:15:02,852: INFO: common: created directory at: artifacts]
[2026-03-13 17:15:02,852: INFO: common: created directory at: artifacts/data_transformation]


AttributeError: 'NoneType' object has no attribute 'endswith'

In [42]:
import os
print(os.getcwd())

d:\


In [45]:
config = ConfigurationManager()
data_transformation_config = config.get_data_transformation_config()

print("Tokenizer:", data_transformation_config.tokenizer_name)

[2026-03-13 17:20:50,459: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-13 17:20:50,463: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-13 17:20:50,465: INFO: common: created directory at: artifacts]
[2026-03-13 17:20:50,467: INFO: common: created directory at: artifacts/data_transformation]
Tokenizer: google/pegasus-cnn_dailymail


In [43]:
import os
os.chdir("D:/Text-Summarization-NLP-Project")
print(os.getcwd())

D:\Text-Summarization-NLP-Project


In [46]:
!pip uninstall transformers tokenizers -y
!pip install transformers tokenizers sentencepiece --upgrade

Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2


You can safely remove it manually.


  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)

   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- ------------------- 1/2 [transformers]
   -------------------- --------------

In [47]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/pegasus-cnn_dailymail")
print(tokenizer)

AttributeError: 'NoneType' object has no attribute 'endswith'